In [1]:
# 1. Install dependencies (safe to re-run)
import sys, subprocess
pkgs = ["huggingface_hub>=0.26.0", "ipywidgets>=8.0", "jsonschema", "python-dotenv"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--break-system-packages", *pkgs], check=True)
print("✅ Dependencies ready.")

✅ Dependencies ready.


In [2]:
import os
from getpass import getpass
from dotenv import load_dotenv
load_dotenv()  # loads .env if present; never prints its contents

class Config:
    MODEL = os.environ.get("HF_MODEL", "meta-llama/Llama-3.1-8B-Instruct") # Changed model
    PROVIDER = os.environ.get("HF_PROVIDER", "auto")   # "auto" lets HF route to any provider that serves MODEL
    TEMPERATURE = float(os.environ.get("LLM_TEMPERATURE", 0.4))
    MAX_TOKENS = int(os.environ.get("LLM_MAX_TOKENS", 2000))
    TIMEOUT_SECONDS = int(os.environ.get("LLM_TIMEOUT", 30))
    MAX_RETRIES = int(os.environ.get("LLM_MAX_RETRIES", 3))
    RETRY_BACKOFF_BASE = 1.5   # seconds, exponential backoff
    API_KEY = None              # set below, in memory only

def _prompt_for_token() -> str:
    while True:
        token = getpass("Enter your Hugging Face API token (input hidden, press Enter to submit): ").strip()
        if token:
            return token
        print("⚠️  Token cannot be empty. Please paste a real token from https://huggingface.co/settings/tokens")

_env_key = os.environ.get("HF_API_KEY") or os.environ.get("HF_TOKEN")
Config.API_KEY = _env_key if _env_key else _prompt_for_token()

def api_key_configured() -> bool:
    return bool(Config.API_KEY)

print("Key present:", api_key_configured(), "| Model:", Config.MODEL, "| Provider:", Config.PROVIDER)

# --- Live verification: one tiny REAL call, no mock. Confirms key + model + provider all actually work. ---
from huggingface_hub import InferenceClient
from huggingface_hub.errors import HfHubHTTPError, InferenceTimeoutError

def verify_connection() -> bool:
    try:
        client = InferenceClient(model=Config.MODEL, token=Config.API_KEY,
                                  provider=Config.PROVIDER, timeout=Config.TIMEOUT_SECONDS)
        resp = client.chat.completions.create(
            model=Config.MODEL,
            messages=[{"role": "user", "content": "Reply with only the word: OK"}],
            max_tokens=5,
        )
        text = (resp.choices[0].message.content or "").strip()
        print(f"✅ Connected. Hugging Face responded: {text!r}")
        return True
    except HfHubHTTPError as e:
        status = getattr(getattr(e, "response", None), "status_code", None)
        if status in (401, 403):
            print("❌ Invalid or unauthorized token. Generate a new 'Read' token and re-run this cell.")
        elif status == 404:
            print(f"❌ Model '{Config.MODEL}' isn't available through Inference Providers. Try a different HF_MODEL.")
        elif status == 429:
            print("⚠️  Rate limited right now, but the key/model look valid. Try again shortly.")
        else:
            print(f"❌ Hugging Face returned an error (status {status}): {e}")
        return False
    except InferenceTimeoutError:
        print("❌ Connection timed out. Check your network and try again.")
        return False
    except Exception as e:
        print(f"❌ Unexpected error verifying the connection: {e}")
        return False

CONNECTION_OK = verify_connection()

Key present: True | Model: meta-llama/Llama-3.1-8B-Instruct | Provider: auto
✅ Connected. Hugging Face responded: 'OK'


## 3. Output schema & validation

In [3]:
# 3. JSON schema for the itinerary response
from jsonschema import validate as jsonschema_validate, ValidationError

ITINERARY_SCHEMA = {
    "type": "object",
    "required": ["place", "budget", "days", "estimated_total_cost", "notes"],
    "properties": {
        "place": {"type": "string", "minLength": 1},
        "budget": {
            "type": "object",
            "required": ["amount", "currency"],
            "properties": {
                "amount": {"type": "number", "minimum": 0},
                "currency": {"type": "string"}
            }
        },
        "days": {
            "type": "array",
            "minItems": 3,
            "maxItems": 3,
            "items": {
                "type": "object",
                "required": ["day", "activities", "estimated_day_cost"],
                "properties": {
                    "day": {"type": "integer", "minimum": 1, "maximum": 3},
                    "estimated_day_cost": {"type": "number", "minimum": 0},
                    "activities": {
                        "type": "array",
                        "minItems": 1,
                        "items": {
                            "type": "object",
                            "required": ["time", "location", "activity", "estimated_cost"],
                            "properties": {
                                "time": {"type": "string"},
                                "location": {"type": "string"},
                                "activity": {"type": "string"},
                                "estimated_cost": {"type": "number", "minimum": 0},
                                "travel_from_previous_minutes": {"type": ["number", "null"]}
                            }
                        }
                    }
                }
            }
        },
        "estimated_total_cost": {"type": "number", "minimum": 0},
        "notes": {"type": "array", "items": {"type": "string"}}
    }
}

class ItineraryValidationError(Exception):
    pass

def validate_itinerary_schema(data: dict) -> None:
    '''Raises ItineraryValidationError with a human-readable reason on failure.'''
    try:
        jsonschema_validate(instance=data, schema=ITINERARY_SCHEMA)
    except ValidationError as e:
        raise ItineraryValidationError(f"Schema validation failed: {e.message}")

    days = data.get("days", [])
    day_numbers = sorted(d.get("day") for d in days)
    if day_numbers != [1, 2, 3]:
        raise ItineraryValidationError(f"Expected days [1, 2, 3], got {day_numbers}")

print("✅ Schema loaded.")

✅ Schema loaded.


## 4. Input validation

In [4]:
# 4. Input validation
from dataclasses import dataclass, field
from typing import Optional, List

MAX_INTERESTS = 8
MAX_NOTE_LEN = 300

class InputValidationError(Exception):
    pass

@dataclass
class TripRequest:
    place: str
    budget_amount: float
    currency: str = "INR"
    interests: List[str] = field(default_factory=list)
    pace: Optional[str] = None
    family_friendly: bool = False
    start_time: Optional[str] = None
    dietary: Optional[str] = None
    notes: Optional[str] = None

def validate_place(place: str) -> str:
    place = (place or "").strip()
    if not place:
        raise InputValidationError("Place is required.")
    if len(place) > 80:
        raise InputValidationError("Place name looks too long — please shorten it.")
    return place

def validate_budget(amount) -> float:
    if isinstance(amount, str):
        amount = amount.replace(",", "").strip()
    try:
        amount = float(amount)
    except (TypeError, ValueError):
        raise InputValidationError("Budget must be a number, e.g. 10000.")
    if amount <= 0:
        raise InputValidationError("Budget must be greater than zero.")
    return amount

def validate_interests(interests) -> List[str]:
    if isinstance(interests, str):
        interests = [i.strip() for i in interests.split(",") if i.strip()]
    if not interests:
        raise InputValidationError("At least one interest is required (e.g. History, Vegan Food).")
    if len(interests) > MAX_INTERESTS:
        raise InputValidationError(f"Please list at most {MAX_INTERESTS} interests.")
    return interests

def validate_pace(pace: Optional[str]) -> Optional[str]:
    if not pace:
        return None
    pace = pace.strip().lower()
    if pace not in {"relaxed", "moderate", "packed"}:
        raise InputValidationError("Pace must be one of: relaxed, moderate, packed.")
    return pace

def build_trip_request(place, budget_amount, currency, interests, pace=None,
                        family_friendly=False, start_time=None, dietary=None, notes=None) -> TripRequest:
    '''Validates all inputs together; raises InputValidationError on the first problem found (FR-04).'''
    place = validate_place(place)
    budget_amount = validate_budget(budget_amount)
    interests = validate_interests(interests)
    pace = validate_pace(pace)
    if notes and len(notes) > MAX_NOTE_LEN:
        raise InputValidationError(f"Notes must be under {MAX_NOTE_LEN} characters.")
    return TripRequest(
        place=place, budget_amount=budget_amount, currency=(currency or "INR").upper(),
        interests=interests, pace=pace, family_friendly=bool(family_friendly),
        start_time=start_time or None, dietary=dietary or None, notes=notes or None,
    )

print("✅ Input validators ready.")

✅ Input validators ready.


## 5. Prompt strategy

In [5]:
# 5. Prompt builder
import json

_SCHEMA_EXAMPLE = {
    "place": "string",
    "budget": {"amount": 0, "currency": "string"},
    "days": [{
        "day": 1,
        "activities": [{
            "time": "HH:MM", "location": "string", "activity": "string",
            "estimated_cost": 0, "travel_from_previous_minutes": 0
        }],
        "estimated_day_cost": 0
    }],
    "estimated_total_cost": 0,
    "notes": ["string"]
}

def build_system_prompt() -> str:
    schema_str = json.dumps(_SCHEMA_EXAMPLE)
    return (
        "You are a practical, budget-aware travel-planning assistant. "
        "You produce a realistic 3-day itinerary for a single place, based only on general knowledge. "
        "Rules you must follow strictly:\n"
        "1. Respond with ONLY a single valid JSON object — no markdown fences, no commentary, no preamble.\n"
        "2. The JSON MUST match this exact shape:\n"
        f"{schema_str}\n"
        "3. Always return EXACTLY 3 day objects, numbered 1, 2, 3.\n"
        "4. All prices and travel times are ESTIMATES ONLY. Never claim live traffic, live opening hours, "
        "live prices, booking availability, or confirmed reservations. Never fabricate bookings.\n"
        "5. Try to keep the total within the stated budget. If that is not realistically possible, still "
        "produce the best plan you can and add a note clearly flagging that the budget may be insufficient.\n"
        "6. Keep activities practical, geographically sensible, and relevant to the traveler's interests."
    )

def build_user_prompt(req: "TripRequest") -> str:
    lines = [
        f"Place: {req.place}",
        f"Budget: {req.budget_amount} {req.currency}",
        f"Interests: {', '.join(req.interests)}",
    ]
    if req.pace:
        lines.append(f"Preferred pace: {req.pace}")
    if req.family_friendly:
        lines.append("Family-friendly activities preferred.")
    if req.start_time:
        lines.append(f"Preferred start time each day: {req.start_time}")
    if req.dietary:
        lines.append(f"Dietary preferences: {req.dietary}")
    if req.notes:
        lines.append(f"Other notes: {req.notes}")
    lines.append("Generate the 3-day itinerary JSON now, following the schema and rules exactly.")
    return "\n".join(lines)

print("✅ Prompt builders ready.")

✅ Prompt builders ready.


## 6. LLM API function

In [6]:
import time
from huggingface_hub import InferenceClient
from huggingface_hub.errors import HfHubHTTPError, InferenceTimeoutError, BadRequestError, OverloadedError
import requests

class LLMCallError(Exception):
    '''Raised for any unrecoverable LLM call failure (after retries are exhausted).'''
    def __init__(self, message, category="unknown"):
        super().__init__(message)
        self.category = category  # "missing_key" | "timeout" | "rate_limit" | "network" | "server" | "bad_model" | "unknown"

_client = None

def _get_client() -> InferenceClient:
    global _client
    if not api_key_configured():
        raise LLMCallError("Missing API key. Re-run the Configuration cell and enter your Hugging Face token.",
                            category="missing_key")
    if _client is None:
        _client = InferenceClient(
            model=Config.MODEL, token=Config.API_KEY,
            provider=Config.PROVIDER, timeout=Config.TIMEOUT_SECONDS,
        )
    return _client

def _status_of(exc: HfHubHTTPError):
    return getattr(getattr(exc, "response", None), "status_code", None)

def call_llm(system_prompt: str, user_prompt: str) -> str:
    '''
    Sends system+user messages to the configured Hugging Face model via the real chat-completion API
    and returns the raw response text. Handles timeouts, rate limits, and transient network errors with
    exponential-backoff retries. Never logs or exposes the API key. No mock/fallback path exists.
    '''
    client = _get_client()
    last_error = None

    for attempt in range(1, Config.MAX_RETRIES + 1):
        try:
            resp = client.chat.completions.create(
                model=Config.MODEL,
                temperature=Config.TEMPERATURE,
                max_tokens=Config.MAX_TOKENS,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
            )
            text = resp.choices[0].message.content
            if not text or not text.strip():
                raise LLMCallError("The AI returned an empty response.", category="unknown")
            return text

        except InferenceTimeoutError:
            last_error = LLMCallError("The AI request timed out.", category="timeout")
        except requests.exceptions.ConnectionError:
            last_error = LLMCallError("Network error while contacting Hugging Face.", category="network")
        except BadRequestError as e:
            last_error = LLMCallError(f"The model rejected the request: {e}", category="bad_model")
            raise last_error  # not retryable — the request itself is malformed for this model
        except OverloadedError:
            last_error = LLMCallError("The model is overloaded right now.", category="server")
        except HfHubHTTPError as e:
            status = _status_of(e)
            if status in (401, 403):
                last_error = LLMCallError("Invalid or unauthorized Hugging Face API key.", category="missing_key")
                raise last_error  # not retryable — bad credentials won't fix themselves
            elif status == 404:
                last_error = LLMCallError(
                    f"Model '{Config.MODEL}' isn't available through Inference Providers.", category="bad_model")
                raise last_error  # not retryable — wrong model id
            elif status == 429:
                last_error = LLMCallError("Rate limit reached, please slow down.", category="rate_limit")
            elif status == 503:
                last_error = LLMCallError("Model is loading on Hugging Face's servers.", category="server")
            elif status and status >= 500:
                last_error = LLMCallError(f"Hugging Face returned a server error (status {status}).", category="server")
            else:
                last_error = LLMCallError(f"Hugging Face returned an error (status {status}): {e}", category="unknown")
        except LLMCallError:
            raise
        except Exception as e:
            last_error = LLMCallError(f"Unexpected error calling the LLM: {e}", category="unknown")

        if attempt < Config.MAX_RETRIES:
            time.sleep(Config.RETRY_BACKOFF_BASE ** attempt)

    raise last_error

print("✅ call_llm() ready (real Hugging Face calls, no mock path).")

✅ call_llm() ready (real Hugging Face calls, no mock path).


## 7. Response parsing, repair & validation

In [ ]:
# 7. Parse, repair, and validate the LLM's response
import json
import re

def _extract_json_block(text: str) -> str:
    text = text.strip()
    fenced = re.search(r"```(?:json)?\s*(\{.*\})\s*```", text, re.DOTALL)
    if fenced:
        return fenced.group(1)
    first, last = text.find("{"), text.rfind("}")
    if first != -1 and last != -1 and last > first:
        return text[first:last + 1]
    return text

def _normalize_itinerary_keys(data: dict) -> dict:
    """Normalize common model variations before strict schema validation."""
    if not isinstance(data, dict):
        return data
    data = dict(data)
    if "place" not in data:
        for alias in ("city", "destination"):
            if alias in data:
                data["place"] = data.pop(alias)
                break

    def number(value):
        if isinstance(value, str):
            cleaned = value.replace(",", "").strip()
            match = re.search(r"-?\d+(?:\.\d+)?", cleaned)
            if match:
                return float(match.group(0))
            if cleaned.lower() in {"free", "none", "n/a", "na"}:
                return 0.0
        return value

    data["estimated_total_cost"] = number(data.get("estimated_total_cost"))
    if isinstance(data.get("budget"), dict):
        data["budget"] = dict(data["budget"])
        data["budget"]["amount"] = number(data["budget"].get("amount"))
    for day in data.get("days", []):
        if isinstance(day, dict):
            day["day"] = number(day.get("day"))
            day["estimated_day_cost"] = number(day.get("estimated_day_cost"))
            for activity in day.get("activities", []):
                if isinstance(activity, dict):
                    activity["estimated_cost"] = number(activity.get("estimated_cost"))
                    if activity.get("travel_from_previous_minutes") is not None:
                        activity["travel_from_previous_minutes"] = number(
                            activity["travel_from_previous_minutes"]
                        )
    return data

def parse_and_validate(raw_text: str) -> dict:
    candidate = _extract_json_block(raw_text)
    try:
        data = json.loads(candidate)
    except json.JSONDecodeError as e:
        raise ItineraryValidationError(f"Could not parse JSON: {e}")
    data = _normalize_itinerary_keys(data)
    validate_itinerary_schema(data)
    return data

def repair_and_parse(raw_text: str, validation_error: str = "") -> dict:
    '''One real repair call to the LLM — no mock substitution.'''
    schema_example = json.dumps(_SCHEMA_EXAMPLE)
    repair_system = (
        "You are a strict JSON repair tool. Return exactly ONE valid JSON object and nothing else. "
        "Use the exact schema example below. The top-level destination field MUST be named 'place', never 'city'. "
        "Return exactly 3 day objects numbered 1, 2, and 3. Every activity must include time, location, "
        "activity, and estimated_cost. travel_from_previous_minutes may be null. "
        "Use numbers for all costs, not strings or currency symbols.\n\n"
        f"Exact schema example:\n{schema_example}"
    )
    repair_user = (
        f"Validation error: {validation_error}\n"
        f"Broken output to repair:\n{raw_text}"
    )
    fixed_text = call_llm(repair_system, repair_user)
    return parse_and_validate(fixed_text)

def get_itinerary_json(req: "TripRequest") -> dict:
    raw = call_llm(build_system_prompt(), build_user_prompt(req))
    try:
        return parse_and_validate(raw)
    except ItineraryValidationError as first_error:
        return repair_and_parse(raw, str(first_error))

print("✅ Parser/validator ready.")

✅ Parser/validator ready.


## 8. Orchestrator



In [ ]:
# 8. End-to-end orchestrator
def generate_itinerary(place, budget_amount, currency, interests, pace=None,
                        family_friendly=False, start_time=None, dietary=None, notes=None) -> dict:
    try:
        req = build_trip_request(place, budget_amount, currency, interests, pace,
                                  family_friendly, start_time, dietary, notes)
    except InputValidationError as e:
        return {"ok": False, "stage": "validation", "error": str(e)}

    try:
        itinerary = get_itinerary_json(req)
    except LLMCallError as e:
        friendly = {
            "missing_key": "Your Hugging Face API key is missing or invalid. Re-run the Configuration cell.",
            "timeout": "The AI took too long to respond. Please try again.",
            "rate_limit": "Too many requests right now. Please wait a moment and try again.",
            "network": "Couldn't reach Hugging Face. Check your connection and try again.",
            "server": "Hugging Face had a server-side issue (the model may still be loading). Try again shortly.",
            "bad_model": f"The configured model ('{Config.MODEL}') can't serve this request. Try a different HF_MODEL.",
            "unknown": "Something went wrong generating your itinerary. Please try again.",
        }.get(e.category, "Something went wrong generating your itinerary.")
        return {"ok": False, "stage": "llm_call", "error": friendly, "category": e.category}
    except ItineraryValidationError as e:
        return {"ok": False, "stage": "validation_of_ai_output",
                "error": f"The AI response did not match the itinerary schema: {e}"}
    except Exception as e:
        return {"ok": False, "stage": "unknown", "error": f"An unexpected error occurred: {e}"}

    over_budget = itinerary.get("estimated_total_cost", 0) > req.budget_amount * 1.15
    return {"ok": True, "itinerary": itinerary, "request": req, "over_budget": over_budget}

print("✅ Orchestrator ready.")

✅ Orchestrator ready.


## 9. Display — professional HTML rendering

In [9]:
# 9. Render the itinerary as a polished HTML card layout
from html import escape

def format_itinerary_html(result: dict) -> str:
    if not result.get("ok"):
        return f'''
        <div style="font-family:-apple-system,Segoe UI,Roboto,sans-serif;border:1px solid #f5c2c7;
                    background:#f8d7da;color:#842029;padding:18px 20px;border-radius:12px;">
            <div style="font-size:17px;font-weight:600;margin-bottom:6px;">⚠️ Couldn't generate your itinerary</div>
            <div style="font-size:13px;opacity:0.8;margin-bottom:8px;">Stage: {escape(str(result.get('stage')))}</div>
            <div style="font-size:14px;">{escape(str(result.get('error')))}</div>
        </div>'''

    it, req = result["itinerary"], result["request"]
    currency = it["budget"]["currency"]
    warn = ""
    if result.get("over_budget"):
        warn = '''<div style="background:#fff3cd;color:#664d03;border:1px solid #ffe69c;padding:10px 14px;
                   border-radius:8px;font-size:13px;margin:12px 0;">⚠️ Estimated total may exceed your budget.</div>'''

    day_cards = ""
    for day in sorted(it["days"], key=lambda d: d["day"]):
        acts = ""
        for a in day["activities"]:
            travel = a.get("travel_from_previous_minutes")
            travel_chip = (f'<span style="background:#e7f1ff;color:#0a58ca;font-size:11px;padding:2px 8px;'
                            f'border-radius:999px;margin-left:6px;">🚗 ~{travel} min</span>') if travel else ""
            acts += f'''
            <div style="display:flex;gap:12px;padding:10px 0;border-bottom:1px solid #eee;">
                <div style="min-width:56px;font-weight:600;color:#495057;font-size:13px;">{escape(a['time'])}</div>
                <div style="flex:1;">
                    <div style="font-weight:600;color:#212529;font-size:14px;">{escape(a['activity'])}</div>
                    <div style="color:#6c757d;font-size:12.5px;margin-top:2px;">📍 {escape(a['location'])}{travel_chip}</div>
                </div>
                <div style="font-weight:600;color:#198754;font-size:13px;white-space:nowrap;">
                    {a['estimated_cost']} {escape(currency)}
                </div>
            </div>'''
        day_cards += f'''
        <div style="background:#fff;border:1px solid #e9ecef;border-radius:14px;padding:18px 20px;margin-bottom:16px;
                    box-shadow:0 1px 3px rgba(0,0,0,0.04);">
            <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:6px;">
                <div style="font-size:16px;font-weight:700;color:#212529;">Day {day['day']}</div>
                <div style="font-size:13px;font-weight:600;color:#495057;background:#f1f3f5;padding:3px 10px;
                            border-radius:999px;">~{day['estimated_day_cost']} {escape(currency)}</div>
            </div>
            {acts}
        </div>'''

    notes_html = ""
    if it.get("notes"):
        items = "".join(f"<li style='margin-bottom:4px;'>{escape(n)}</li>" for n in it["notes"])
        notes_html = f'''<div style="background:#f8f9fa;border-radius:10px;padding:14px 18px;margin-top:8px;
                          font-size:13px;color:#495057;"><b>Notes</b><ul style="margin:6px 0 0 18px;padding:0;">
                          {items}</ul></div>'''

    return f'''
    <div style="font-family:-apple-system,Segoe UI,Roboto,sans-serif;max-width:640px;">
        <div style="background:linear-gradient(135deg,#4f46e5,#0ea5e9);border-radius:16px;padding:22px 24px;
                    color:white;margin-bottom:18px;">
            <div style="font-size:20px;font-weight:700;">🗺️ 3-Day Itinerary — {escape(it['place'])}</div>
            <div style="font-size:13px;opacity:0.92;margin-top:6px;">
                Budget: {it['budget']['amount']} {escape(currency)} &nbsp;•&nbsp; Interests: {escape(', '.join(req.interests))}
            </div>
        </div>
        {warn}
        {day_cards}
        <div style="display:flex;justify-content:space-between;align-items:center;background:#212529;color:white;
                    border-radius:12px;padding:14px 20px;margin-top:4px;">
            <span style="font-size:14px;">Estimated total</span>
            <span style="font-size:18px;font-weight:700;">{it['estimated_total_cost']} {escape(currency)}</span>
        </div>
        {notes_html}
        <div style="font-size:11.5px;color:#adb5bd;margin-top:10px;text-align:center;">
            All prices and travel times are estimates — not live data or confirmed bookings.
        </div>
    </div>'''

print("✅ Display formatter ready.")

✅ Display formatter ready.


## 10. Chatbox UI

In [10]:
# 10. Professional chatbox UI
import ipywidgets as w
from IPython.display import display, HTML, clear_output

FIELD_LAYOUT = w.Layout(width="100%")

header = w.HTML('''
<div style="font-family:-apple-system,Segoe UI,Roboto,sans-serif;background:linear-gradient(135deg,#4f46e5,#0ea5e9);
            border-radius:16px 16px 0 0;padding:20px 24px;color:white;">
    <div style="font-size:19px;font-weight:700;">✈️ Plan your trip</div>
    <div style="font-size:12.5px;opacity:0.9;margin-top:3px;">Enter your details and press Enter (or Generate) — powered by Hugging Face AI</div>
</div>''')

status_dot = "🟢" if api_key_configured() and CONNECTION_OK else "🔴"
status_text = "Connected to Hugging Face" if (api_key_configured() and CONNECTION_OK) else "Not connected — re-run the Configuration cell"
status_bar = w.HTML(f'''<div style="font-family:-apple-system,Segoe UI,Roboto,sans-serif;font-size:12px;
                        color:#495057;padding:8px 24px;background:#f8f9fa;border-bottom:1px solid #e9ecef;">
                        {status_dot} {status_text} &nbsp;|&nbsp; Model: {escape(Config.MODEL)}</div>''')

place_box = w.Text(placeholder="e.g. Chennai", description="Place", layout=FIELD_LAYOUT, style={"description_width": "90px"})
budget_box = w.Text(placeholder="e.g. 10000", description="Budget", layout=FIELD_LAYOUT, style={"description_width": "90px"})
currency_box = w.Text(value="INR", description="Currency", layout=FIELD_LAYOUT, style={"description_width": "90px"})
interests_box = w.Text(placeholder="e.g. History, Vegan Food", description="Interests", layout=FIELD_LAYOUT, style={"description_width": "90px"})
pace_box = w.Dropdown(options=["", "relaxed", "moderate", "packed"], description="Pace", layout=FIELD_LAYOUT, style={"description_width": "90px"})
family_box = w.Checkbox(value=False, description="Family-friendly")
start_time_box = w.Text(placeholder="e.g. 09:00", description="Start time", layout=FIELD_LAYOUT, style={"description_width": "90px"})
dietary_box = w.Text(placeholder="e.g. vegan", description="Dietary", layout=FIELD_LAYOUT, style={"description_width": "90px"})
notes_box = w.Textarea(placeholder="Any other preferences...", description="Notes", layout=w.Layout(width="100%", height="60px"), style={"description_width": "90px"})

generate_btn = w.Button(description="✨ Generate Itinerary", button_style="success", layout=w.Layout(width="220px"))
regenerate_btn = w.Button(description="🔁 Regenerate", button_style="info", layout=w.Layout(width="160px"), disabled=True)
clear_btn = w.Button(description="Clear", layout=w.Layout(width="100px"))
error_label = w.HTML("")
output_area = w.Output()

def _set_error(msg: str):
    error_label.value = f'<div style="color:#b02a37;font-size:12.5px;margin-top:4px;">{escape(msg)}</div>' if msg else ""

def _set_busy(is_busy: bool):
    generate_btn.disabled = is_busy
    regenerate_btn.disabled = is_busy or regenerate_btn.disabled
    clear_btn.disabled = is_busy
    generate_btn.description = "⏳ Generating..." if is_busy else "✨ Generate Itinerary"

def _run_generation(_=None):
    if not api_key_configured():
        _set_error("No Hugging Face API key is configured. Re-run the Configuration cell above.")
        return
    _set_error("")
    _set_busy(True)
    with output_area:
        clear_output()
        display(HTML('<div style="font-family:sans-serif;color:#495057;font-size:13px;">⏳ Calling the real Hugging Face model — this can take a few seconds...</div>'))
    try:
        result = generate_itinerary(
            place=place_box.value, budget_amount=budget_box.value, currency=currency_box.value,
            interests=interests_box.value, pace=pace_box.value or None,
            family_friendly=family_box.value, start_time=start_time_box.value or None,
            dietary=dietary_box.value or None, notes=notes_box.value or None,
        )
    finally:
        _set_busy(False)

    with output_area:
        clear_output()
        display(HTML(format_itinerary_html(result)))
    if result.get("ok"):
        regenerate_btn.disabled = False

generate_btn.on_click(_run_generation)
regenerate_btn.on_click(_run_generation)

def _clear_form(_=None):
    for box in (place_box, budget_box, interests_box, start_time_box, dietary_box):
        box.value = ""
    currency_box.value = "INR"
    pace_box.value = ""
    family_box.value = False
    notes_box.value = ""
    _set_error("")
    with output_area:
        clear_output()

clear_btn.on_click(_clear_form)

# Enter key support: every single-line Text field submits the form on Enter.
for _box in (place_box, budget_box, currency_box, interests_box, start_time_box, dietary_box):
    _box.on_submit(_run_generation)

form_body = w.VBox([
    w.HBox([place_box, currency_box]),
    w.HBox([budget_box, pace_box]),
    interests_box,
    w.HBox([family_box, start_time_box, dietary_box]),
    notes_box,
    error_label,
    w.HBox([generate_btn, regenerate_btn, clear_btn]),
], layout=w.Layout(padding="18px 24px"))

card = w.VBox([header, status_bar, form_body, output_area],
              layout=w.Layout(border="1px solid #e9ecef", border_radius="16px", width="720px",
                               box_shadow="0 2px 10px rgba(0,0,0,0.06)"))
display(card)

C:\Users\siddi\AppData\Local\Temp\ipykernel_17564\2274575043.py:88: DeprecationWarning: on_submit is deprecated. Instead, set the .continuous_update attribute to False and observe the value changing with: mywidget.observe(callback, 'value').
  _box.on_submit(_run_generation)
